# Задание 1. Посмотрим на объект модели

In [11]:
import joblib
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [7]:
load_model_from_internet = False
model_name = "t5-base"

if load_model_from_internet:
    # загрузка модели и токенизатора
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    joblib.dump(model, '../models/' + model_name + '_model_08.joblib')
    joblib.dump(tokenizer, '../models/' + model_name + '_tokenizer_08.joblib')
else:
    model = joblib.load('../models/' + model_name + '_model_08.joblib')
    tokenizer = joblib.load('../models/' + model_name + '_tokenizer_08.joblib')

In [8]:

# 1) Общая конфигурация
cfg = model.config
print("Model config:")
print(f"d_model={cfg.d_model}, num_layers={cfg.num_layers}, feed_forward_size={cfg.d_ff}")
print()

# 2) Блок энкодера
first_encoder_block = model.encoder.block[0]
print("Encoder block:")
print(first_encoder_block)
print()

# 3) Декодер — блоки с self-attention и cross-attention
first_decoder_block = model.decoder.block[0]
print("Decoder block:")
print(first_decoder_block)
print()

# 4) количество параметров
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Model config:
d_model=768, num_layers=12, feed_forward_size=3072

Encoder block:
T5Block(
  (layer): ModuleList(
    (0): T5LayerSelfAttention(
      (SelfAttention): T5Attention(
        (q): Linear(in_features=768, out_features=768, bias=False)
        (k): Linear(in_features=768, out_features=768, bias=False)
        (v): Linear(in_features=768, out_features=768, bias=False)
        (o): Linear(in_features=768, out_features=768, bias=False)
        (relative_attention_bias): Embedding(32, 12)
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): T5LayerFF(
      (DenseReluDense): T5DenseActDense(
        (wi): Linear(in_features=768, out_features=3072, bias=False)
        (wo): Linear(in_features=3072, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (act): ReLU()
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)

Decoder block:
T5Block(
  (layer

# Задание 2. Подготовка датасета
* Для этого в каждом тексте надо обрезать с краёв пробелы, если они есть, заменить перенос строки на пробел и добавить инструкцию `summarize:`, а затем токенизировать текст. 
* Токенизированные тексты надо сохранить в список `tokenized_texts`.
* Инструкцию для суммаризации можно добавить так: input_text = "summarize: " + text.
* Чтобы токенизировать текст, передайте его в объект класса `tokenizer` с параметрами:
  * `return_tensors="pt",`
  * `truncation=True,`
  * `max_length=512,  # T5-base ограничен 512 токенами`
  * `padding="max_length"`

In [ ]:
import torch
from datasets import load_dataset


README.md: 0.00B [00:00, ?B/s]

c:\Users\aseva\Desktop\MyEDU\YaDLE\YaDLE_project_vscode_stream_2\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aseva\.cache\huggingface\hub\datasets--billsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling 

train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

...

ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

Example from dataset:
{'text': 'The people of the State of California do enact as follows:\n\n\nSECTION 1.\nThe Legislature finds and declares all of the following:\n(a) (1) Since 1899 congressionally chartered veterans’ organizations have provided a valuable service to our nation’s returning service members. These organizations help preserve the memories and incidents of the great hostilities fought by our nation, and preserve and strengthen comradeship among members.\n(2) These veterans’ organizations also own and manage various properties including lodges, posts, and fraternal halls. These properties act as a safe haven where veterans of all ages and their families can gather together to find camaraderie and fellowship, share stories, and seek support from people who understand their unique experiences. This aids in the healing process for these returning veterans, and ensures their health and happiness.\n(b) As a result of congressional chartering of these veterans’ organizations, 

In [18]:
load_view_dataset = False

if load_view_dataset:
    # Загружаем подмножество для быстрого теста
    dataset = load_dataset("billsum", split="ca_test[:50]")

    pd.DataFrame(
                 index=['text','summary','title'],
                 data=[
                       dataset['text'],
                       dataset['summary'],
                       dataset['title']
                      ]
    ).T.to_csv('../data/bill_summary_first50.csv', sep=';', index=False)
else:
    dataset = pd.read_csv ('../data/bill_summary_first50.csv', sep=';')


In [ ]:

# Пример
# тексты и их референсные суммаризации

print("Example from dataset:")
if load_view_dataset:
    print(dataset[0])

    texts = [item['text'] for item in dataset]
    references = [item['summary'] for item in dataset]

else:
    print(dataset.iloc[0])
    
    texts = [item for item in dataset['text']]
    references = [item for item in dataset['summary']]
  

tokenized_texts = []

for text in texts:
    input_text = "summarize: " + text.strip().replace("\n", " ")
    tokenized_input_text = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512,  # T5-base ограничен 512 токенами
        padding="max_length"
    )

    tokenized_texts.append(tokenized_input_text)

print('Example of tokenized text:')
print(tokenized_texts[0]['input_ids'])

Example from dataset:
text       The people of the State of California do enact...
summary    Existing property tax law establishes a vetera...
title      An act to amend Section 215.1 of the Revenue a...
Name: 0, dtype: object
Example of tokenized text:
tensor([[21603,    10,    37,   151,    13,     8,  1015,    13,  1826,   103,
             3,    35,  2708,    38,  6963,    10,   180,  3073,  9562,  1300,
            37, 28204, 12902,    11, 15884,     7,    66,    13,     8,   826,
            10,    41,     9,    61,  5637,  1541,   507,  3264, 28167,   120,
          5059,  3737, 13391,    22,  2371,    43,   937,     3,     9,  3435,
           313,    12,    69,  2982,    22,     7,  7646,   313,   724,     5,
           506,  2371,   199,  8996,     8,  5655,    11, 15935,    13,     8,
           248,  2290,   173,  2197,     3, 13973,    57,    69,  2982,     6,
            11,  8996,    11,  8726,     3,   287, 15530,  2009,   859,   724,
             5,  6499,   506, 1339

# Задание 3. Суммаризация и замер качества
* Теперь сгенерируйте саммари для токенизированных текстов и посчитайте метрику rouge.
* Для генерации саммари воспользуйтесь методом `model.generate()` так же, как это было сделано в теоретической части урока.
* Декодировать сгенерированные токены в текст можно методом `tokenizer.decode()`.
* Создать объект для расчёта метрики можно так: `evaluate.load("rouge")`.
* Для расчёта метрики воспользуйтесь методом `rouge.compute()`.

In [22]:
import evaluate
from tqdm import tqdm

generated_summaries = []

for inputs in tqdm(tokenized_texts):
    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    generated_summaries.append(summary)

rouge = evaluate.load("rouge")

# Подсчёт метрик
results = rouge.compute(predictions=generated_summaries, references=references)
print('Metrics')
for k, v in results.items():
    print(f"{k}: {v:.4f}")

# пример сгенерированного саммари
print('Summary example:')
print(generated_summaries[0])

100%|██████████| 50/50 [04:35<00:00,  5.51s/it]


Metrics
rouge1: 0.1545
rouge2: 0.0571
rougeL: 0.1133
rougeLsum: 0.1299
Summary example:
the state of california enacts a special tax exemption for veterans’ organizations . the exemption applies to all buildings used exclusively for charitable purposes . the state board of equalization concludes that veterans’ organizations are exempt from taxation .
